In [ ]:
import os
import sys
from pathlib import Path
from typing import Dict, Optional, Tuple
from langchain_core.messages import HumanMessage, SystemMessage

PROJECT_ROOT = os.getcwd()
SKILLS_DIR   = os.path.join(PROJECT_ROOT , "../skills")
AGENT_FILE   = os.path.join(PROJECT_ROOT , "../agent/skill_agent.py")

print(f"Project root: {PROJECT_ROOT}")
print(f"Skills directory: {SKILLS_DIR}")
print(f"Agent file: {AGENT_FILE}")

In [ ]:
from pydantic import BaseModel,Field
from typing import Dict, Optional
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate , PromptTemplate

In [ ]:
sys.path.append(os.path.dirname(PROJECT_ROOT))
from handler.skills_registry import load_skill_registry
from agent.skill_agent import list_available_skills, web_page_scraper_tool, read_skill_instructions
from handler.dynamic_skillcreator import create_skill_programmatic 

In [ ]:
os.environ["GROQ_API_KEY"] = 'gsk_'
llm = ChatGroq(model="llama-3.1-8b-instant")

In [ ]:
desc = """
SAM 3: Segment Anything with Concepts
Meta Superintelligence Labs

Nicolas Carion*, Laura Gustafson*, Yuan-Ting Hu*, Shoubhik Debnath*, Ronghang Hu*, Didac Suris*, Chaitanya Ryali*, Kalyan Vasudev Alwala*, Haitham Khedr*, Andrew Huang, Jie Lei, Tengyu Ma, Baishan Guo, Arpit Kalla, Markus Marks, Joseph Greer, Meng Wang, Peize Sun, Roman Rädle, Triantafyllos Afouras, Effrosyni Mavroudi, Katherine Xu°, Tsung-Han Wu°, Yu Zhou°, Liliane Momeni°, Rishi Hazra°, Shuangrui Ding°, Sagar Vaze°, Francois Porcher°, Feng Li°, Siyuan Li°, Aishwarya Kamath°, Ho Kei Cheng°, Piotr Dollar†, Nikhila Ravi†, Kate Saenko†, Pengchuan Zhang†, Christoph Feichtenhofer†

* core contributor, ° intern, † project lead, order is random within groups

[Paper] [Project] [Demo] [Blog] [BibTeX]

SAM 3 architecture SAM 3 is a unified foundation model for promptable segmentation in images and videos. It can detect, segment, and track objects using text or visual prompts such as points, boxes, and masks. Compared to its predecessor SAM 2, SAM 3 introduces the ability to exhaustively segment all instances of an open-vocabulary concept specified by a short text phrase or exemplars. Unlike prior work, SAM 3 can handle a vastly larger set of open-vocabulary prompts. It achieves 75-80% of human performance on our new SA-CO benchmark which contains 270K unique concepts, over 50 times more than existing benchmarks.

This breakthrough is driven by an innovative data engine that has automatically annotated over 4 million unique concepts, creating the largest high-quality open-vocabulary segmentation dataset to date. In addition, SAM 3 introduces a new model architecture featuring a presence token that improves discrimination between closely related text prompts (e.g., “a player in white” vs. “a player in red”), as well as a decoupled detector–tracker design that minimizes task interference and scales efficiently with data.

 

Latest updates
03/27/2026 -- SAM 3.1 Object Multiplex is released. It introduces a shared-memory approach for joint multi-object tracking that is significantly faster without sacrificing accuracy.

A new suite of improved model checkpoints (denoted as SAM 3.1) are released on Hugging Face. See RELEASE_SAM3p1.md for full details.
To use the new SAM 3.1 checkpoints, you need the latest model code from this repo. If you have installed an earlier version of this repo, pull the latest code from this repo (with git pull), and then reinstall the repo following Installation below.
Installation
Prerequisites
Python 3.12 or higher
PyTorch 2.7 or higher
CUDA-compatible GPU with CUDA 12.6 or higher
Create a new Conda environment:
conda create -n sam3 python=3.12
conda deactivate
conda activate sam3
Install PyTorch with CUDA support:
pip install torch==2.10.0 torchvision --index-url https://download.pytorch.org/whl/cu128
Clone the repository and install the package:
git clone https://github.com/facebookresearch/sam3.git
cd sam3
pip install -e .
Install additional dependencies for example notebooks or development:
# For running example notebooks
pip install -e ".[notebooks]"

# For development
pip install -e ".[train,dev]"
Optional dependencies for faster inference
pip install einops ninja && pip install flash-attn-3 --no-deps --index-url https://download.pytorch.org/whl/cu128
pip install git+https://github.com/ronghanghu/cc_torch.git
Getting Started
⚠️ Before using SAM 3, please request access to the checkpoints on the SAM 3 Hugging Face repo. Once accepted, you need to be authenticated to download the checkpoints. You can do this by running the following steps (e.g. hf auth login after generating an access token.)

Basic Usage
import torch
#################################### For Image ####################################
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
# Load the model
model = build_sam3_image_model()
processor = Sam3Processor(model)
# Load an image
image = Image.open("<YOUR_IMAGE_PATH.jpg>")
inference_state = processor.set_image(image)
# Prompt the model with text
output = processor.set_text_prompt(state=inference_state, prompt="<YOUR_TEXT_PROMPT>")

# Get the masks, bounding boxes, and scores
masks, boxes, scores = output["masks"], output["boxes"], output["scores"]

#################################### For Video ####################################

from sam3.model_builder import build_sam3_video_predictor

video_predictor = build_sam3_video_predictor()
video_path = "<YOUR_VIDEO_PATH>" # a JPEG folder or an MP4 video file
# Start a session
response = video_predictor.handle_request(
    request=dict(
        type="start_session",
        resource_path=video_path,
    )
)
response = video_predictor.handle_request(
    request=dict(
        type="add_prompt",
        session_id=response["session_id"],
        frame_index=0, # Arbitrary frame index
        text="<YOUR_TEXT_PROMPT>",
    )
)
output = response["outputs"]
Examples
The examples directory contains notebooks demonstrating how to use SAM3 with various types of prompts:

sam3_image_predictor_example.ipynb : Demonstrates how to prompt SAM 3 with text and visual box prompts on images.
sam3_video_predictor_example.ipynb : Demonstrates how to prompt SAM 3 with text prompts on videos, and doing further interactive refinements with points.
sam3_image_batched_inference.ipynb : Demonstrates how to run batched inference with SAM 3 on images.
sam3_agent.ipynb: Demonsterates the use of SAM 3 Agent to segment complex text prompt on images.
saco_gold_silver_vis_example.ipynb : Shows a few examples from SA-Co image evaluation set.
saco_veval_vis_example.ipynb : Shows a few examples from SA-Co video evaluation set.
There are additional notebooks in the examples directory that demonstrate how to use SAM 3 for interactive instance segmentation in images and videos (SAM 1/2 tasks), or as a tool for an MLLM, and how to run evaluations on the SA-Co dataset.

To run the Jupyter notebook examples:

# Make sure you have the notebooks dependencies installed
pip install -e ".[notebooks]"

# Start Jupyter notebook
jupyter notebook examples/sam3_image_predictor_example.ipynb
Model
SAM 3 consists of a detector and a tracker that share a vision encoder. It has 848M parameters. The detector is a DETR-based model conditioned on text, geometry, and image exemplars. The tracker inherits the SAM 2 transformer encoder-decoder architecture, supporting video segmentation and interactive refinement.

Image Results
Model	Instance Segmentation	Box Detection
LVIS	SA-Co/Gold	LVIS	COCO	SA-Co/Gold
cgF1	AP	cgF1	cgF1	AP	AP	APo	cgF1
Human	-	-	72.8	-	-	-	-	74.0
OWLv2*	29.3	43.4	24.6	30.2	45.5	46.1	23.9	24.5
DINO-X	-	38.5	21.3	-	52.4	56.0	-	22.5
Gemini 2.5	13.4	-	13.0	16.1	-	-	-	14.4
SAM 3	37.2	48.5	54.1	40.6	53.6	56.4	55.7	55.7
* Partially trained on LVIS, APo refers to COCO-O accuracy

Video Results
Model	SA-V test	YT-Temporal-1B test	SmartGlasses test	LVVIS test	BURST test
cgF1	pHOTA	cgF1	pHOTA	cgF1	pHOTA	mAP	HOTA
Human	53.1	70.5	71.2	78.4	58.5	72.3	-	-
SAM 3	30.3	58.0	50.8	69.9	36.4	63.6	36.3	44.5
SA-Co Dataset
We release 2 image benchmarks, SA-Co/Gold and SA-Co/Silver, and a video benchmark SA-Co/VEval. The datasets contain images (or videos) with annotated noun phrases. Each image/video and noun phrase pair is annotated with instance masks and unique IDs of each object matching the phrase. Phrases that have no matching objects (negative prompts) have no masks, shown in red font in the figure. See the linked READMEs for more details on how to download and run evaluations on the datasets.

HuggingFace host: SA-Co/Gold, SA-Co/Silver and SA-Co/VEval
Roboflow host: SA-Co/Gold, SA-Co/Silver and SA-Co/VEval
SA-Co dataset

Development
To set up the development environment:

pip install -e ".[dev,train]"
To format the code:

ufmt format .
Contributing
See contributing and the code of conduct.

License
This project is licensed under the SAM License - see the LICENSE file for details.
"""

In [ ]:
create_skill_programmatic(llm,desc.strip(), log=print,source_hint="SAM 3 Paper Abstract")